# Türkçe sentetik ses verisi üretimi (Chatterbox, Emre'nin sesi)

Bu notebook, Common Voice'un **CC0** Türkçe cümlelerini Chatterbox Multilingual ile Emre'nin sesiyle seslendirir,
Whisper ile kontrol edip hatalı kayıtları eler ve sonucu Piper/VITS eğitimine hazır **LJSpeech** formatında kaydeder.

**Kaggle'da bir kez yapılacaklar**
1. Sağdaki panelde *Settings → Accelerator → GPU T4 x2* seç.
2. *Settings → Internet → On* (telefon doğrulaması gerekiyor).
3. *Add Input → Upload* ile `referans_1.wav` dosyasını yükle (dataset adı fark etmez).
4. *Save Version → Save & Run All (Commit)* de (hızlı deneme için önce `N_CUMLE = 20` ile *Run All*);
   tarayıcıyı kapatsan bile arka planda çalışır ve çıktılar *Output* sekmesinde kalır.

Ayarlar Emre'nin seçtiği gibi: exaggeration 0.5, cfg 0.5, temperature 0.8.

In [ ]:
# Kurulum (Kaggle'ın hazır torch'u yerine Chatterbox'ın istediği sürüm kurulur, 2-4 dk sürer)
# Kurulumdan sonra çekirdeği yeniden başlatmaya gerek yok, ama hata alırsan Run → Restart & Run All yap
!pip install -q chatterbox-tts==0.1.7 "setuptools<81" 2>&1 | grep -v "dependency conflicts\|incompatible" | tail -3
# Kaggle'daki torchvision torch 2.6 ile uyumsuz; transformers onu görünce çöküyor. Uyumlu sürümü temiz kur.
!pip install -q --force-reinstall --no-deps torchvision==0.21.0
# Kaggle'daki torchcodec de yeni torch için derlenmiş ve FFmpeg arıyor; ses dosyalarını librosa ile okuduğumuz için gerekmez
!pip uninstall -y -q torchcodec
import torch; print("torch", torch.__version__, "| GPU sayısı:", torch.cuda.device_count())

In [ ]:
# --- Ayarlar ---
N_CUMLE      = 3000      # tam üretim; hızlı deneme için 20 yap
EXAGGERATION = 0.5
CFG_WEIGHT   = 0.5
TEMPERATURE  = 0.8
CER_ESIGI    = 0.15      # Whisper karakter hata oranı bundan büyükse kayıt elenir
HEDEF_SR     = 22050     # Piper "medium" kalitesi 22050 Hz kullanır
TOHUM        = 1953

import glob, os
REF = next(iter(sorted(glob.glob("/kaggle/input/**/referans*.wav", recursive=True))), None) or os.environ.get("REF_WAV")
assert REF, "referans_1.wav bulunamadı: Add Input → Upload ile yükle"
CALISMA = os.environ.get("CALISMA_DIZINI", "/kaggle/working")
HAM, WAVS = f"{CALISMA}/ham_24k", f"{CALISMA}/emre_tr/wavs"
os.makedirs(HAM, exist_ok=True); os.makedirs(WAVS, exist_ok=True)
print("Referans:", REF)

In [ ]:
# --- Cümleleri hazırla: Common Voice Türkçe metinleri (CC0), sabit commit ---
import random, re, urllib.request
CV = "https://raw.githubusercontent.com/common-voice/common-voice/7a88506c8a0ffbffb0ba5e8117d7975c5fef7c36/server/data/tr/"
satirlar = []
for ad in ["wiki.tr.txt", "sentence-collector.txt", "setimes.txt"]:
    satirlar += urllib.request.urlopen(CV + ad).read().decode("utf-8").splitlines()
izinli = re.compile(r"^[A-Za-zÇĞİÖŞÜçğıöşüâîûÂÎÛ0-9 ,.;:!?'’\-\"()]+$")
temiz = sorted({s.strip() for s in satirlar
                if 25 <= len(s.strip()) <= 180 and izinli.match(s.strip())
                and not re.search(r"\d", s) and s.strip()[-1] in ".!?"})
random.Random(TOHUM).shuffle(temiz)
# Tonlama çeşitliliği için soru ve ünlem cümlelerine de yer ver (yaklaşık %10)
soru = [s for s in temiz if s[-1] in "?!"]
duz = [s for s in temiz if s[-1] == "."]
n_soru = min(len(soru), N_CUMLE // 10)
cumleler = soru[:n_soru] + duz[:N_CUMLE - n_soru]
random.Random(TOHUM).shuffle(cumleler)
cumleler = [(f"emre_{i:05d}", s) for i, s in enumerate(cumleler)]
with open(f"{CALISMA}/cumleler.tsv", "w") as f:
    f.writelines(f"{k}\t{s}\n" for k, s in cumleler)
print(f"{len(temiz)} uygun cümle arasından {len(cumleler)} seçildi. Örnekler:")
for _, s in cumleler[:3]: print(" -", s)

In [ ]:
# --- Üretim: her GPU için ayrı bir işlem, yarıda kalırsa kaldığı yerden devam eder ---
ISCI = r"""
import sys, zlib, torch, torchaudio as ta
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
cihaz, parca, toplam, tsv, ham, ref, ex, cfg, temp = sys.argv[1:]
parca, toplam = int(parca), int(toplam)
isler = [l.rstrip("\n").split("\t") for l in open(tsv)][parca::toplam]
m = ChatterboxMultilingualTTS.from_pretrained(device=cihaz)
m.prepare_conditionals(ref, exaggeration=float(ex))
import os
for i, (k, s) in enumerate(isler):
    yol = f"{ham}/{k}.wav"
    if os.path.exists(yol):
        continue
    torch.manual_seed(zlib.crc32(k.encode()))
    w = m.generate(s, language_id="tr", exaggeration=float(ex), cfg_weight=float(cfg), temperature=float(temp))
    ta.save(yol + ".tmp.wav", w.cpu(), m.sr); os.replace(yol + ".tmp.wav", yol)
    if i % 25 == 0:
        print(f"[{cihaz}] {i+1}/{len(isler)}", flush=True)
print(f"[{cihaz}] bitti", flush=True)
"""
import subprocess, sys, time
open(f"{CALISMA}/isci.py", "w").write(ISCI)
cihazlar = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
os.makedirs(os.path.expanduser("~/.pkuseg"), exist_ok=True)  # iki işlem aynı anda oluşturmaya çalışınca çakışıyor
t0 = time.time()
# Her işlemin çıktısı kendi log dosyasına gider (boru dolup işlem kilitlenmesin); ilerlemeyi dosya sayısından izleriz.
loglar = [open(f"{CALISMA}/isci_{i}.log", "w") for i in range(len(cihazlar))]
islemler = [subprocess.Popen([sys.executable, f"{CALISMA}/isci.py", c, str(i), str(len(cihazlar)),
                              f"{CALISMA}/cumleler.tsv", HAM, REF, str(EXAGGERATION), str(CFG_WEIGHT), str(TEMPERATURE)],
                             stdout=loglar[i], stderr=subprocess.STDOUT, env={**os.environ, "TQDM_DISABLE": "1"})
            for i, c in enumerate(cihazlar)]
while any(p.poll() is None for p in islemler):
    time.sleep(60)
    n = len(glob.glob(HAM + "/*.wav"))
    print(f"{n}/{len(cumleler)} cümle üretildi, {(time.time()-t0)/60:.0f} dk", flush=True)
for i, p in enumerate(islemler):
    if p.returncode:
        print(f"[{cihazlar[i]}] hata ile bitti, log sonu:\n" + "".join(open(f"{CALISMA}/isci_{i}.log").readlines()[-15:]))
print(f"Üretim bitti: {len(glob.glob(HAM + '/*.wav'))} dosya, {(time.time()-t0)/60:.1f} dk")

In [ ]:
# --- Kontrol: Whisper ile yazıya dök, hatalı kayıtları ele, LJSpeech formatında kaydet ---
# Ayrı bir işlemde çalışır; böylece çekirdeği yeniden başlatmadan da temiz bir ortamda açılır.
KONTROL = r"""
import os, re, sys, torch, librosa, soundfile as sf, numpy as np
from transformers import pipeline
calisma, ham, wavs, esik, sr = sys.argv[1], sys.argv[2], sys.argv[3], float(sys.argv[4]), int(sys.argv[5])
cumleler = [l.rstrip("\n").split("\t") for l in open(f"{calisma}/cumleler.tsv")]
gpu = torch.cuda.is_available()
asr = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3-turbo",
               device=0 if gpu else -1, dtype=torch.float16 if gpu else torch.float32)

def sade(s):
    s = s.replace("I", "ı").replace("İ", "i").lower()
    return re.sub(r"[^a-zçğıöşüâîû ]", "", s.replace("’", "").replace("'", "")).split()

def cer(a, b):
    a, b = " ".join(sade(a)), " ".join(sade(b))
    onceki = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        simdi = [i]
        for j, cb in enumerate(b, 1):
            simdi.append(min(onceki[j] + 1, simdi[j-1] + 1, onceki[j-1] + (ca != cb)))
        onceki = simdi
    return onceki[-1] / max(len(a), 1)

kabul, red, toplam_sn = [], [], 0.0
for n, (k, s) in enumerate(cumleler):
    yol = f"{ham}/{k}.wav"
    if not os.path.exists(yol):
        continue
    duyulan = asr(librosa.load(yol, sr=16000)[0], generate_kwargs={"language": "turkish", "task": "transcribe"})["text"].strip()
    hata = cer(s, duyulan)
    if hata > esik:
        red.append((k, round(hata, 3), s, duyulan)); continue
    w, _ = librosa.load(yol, sr=sr)
    w, _ = librosa.effects.trim(w, top_db=40)
    w = np.concatenate([np.zeros(int(0.1*sr)), w, np.zeros(int(0.15*sr))])
    w = 0.95 * w / max(np.abs(w).max(), 1e-6)
    sf.write(f"{wavs}/{k}.wav", w, sr, subtype="PCM_16")
    kabul.append((k, s)); toplam_sn += len(w) / sr
    if n % 200 == 0:
        print(f"kontrol {n+1}/{len(cumleler)}", flush=True)

with open(f"{calisma}/emre_tr/metadata.csv", "w") as f:
    f.writelines(f"{k}|{s}\n" for k, s in kabul)
with open(f"{calisma}/elenenler.tsv", "w") as f:
    f.writelines("\t".join(map(str, r)) + "\n" for r in red)
print(f"Kabul: {len(kabul)}  Elenen: {len(red)}  Toplam süre: {toplam_sn/3600:.2f} saat")
for r in red[:5]:
    print("ELENDİ", r)
"""
open(f"{CALISMA}/kontrol.py", "w").write(KONTROL)
p = subprocess.run([sys.executable, f"{CALISMA}/kontrol.py", CALISMA, HAM, WAVS, str(CER_ESIGI), str(HEDEF_SR)],
                   capture_output=True, text=True)
print(p.stdout)
open(f"{CALISMA}/kontrol_hata.log", "w").write(p.stderr)
if p.returncode:
    raise RuntimeError("Kontrol adımı başarısız. Son satırlar:\n" + "\n".join(p.stderr.splitlines()[-25:]))
kabul = [l.rstrip("\n").split("|", 1) for l in open(f"{CALISMA}/emre_tr/metadata.csv")]

In [ ]:
# --- Birkaç örnek dinle ---
from IPython.display import Audio, display
for k, s in kabul[:3]:
    print(s); display(Audio(f"{WAVS}/{k}.wav"))

In [ ]:
# --- Paketle: çıktı Output sekmesinde emre_tr.zip olarak görünür; eğitim notebook'una girdi olarak eklenir ---
import shutil
shutil.make_archive(f"{CALISMA}/emre_tr", "zip", CALISMA, "emre_tr")
shutil.rmtree(HAM, ignore_errors=True)  # ham 24 kHz dosyalar zip'e girmez, yer kaplamasın
print("Hazır:", f"{CALISMA}/emre_tr.zip")